In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit
from abc import ABC, abstractmethod
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ============================================================
# 1. CLASE ABSTRACTA
# ============================================================
class SistemaFisico(ABC):
    @abstractmethod
    def derivada(self, t, estado):
        """Retorna [dx/dt, dy/dt, dvx/dt, dvy/dt]"""
        pass

    @abstractmethod
    def simular(self, t_span, estado_inicial, t_eval=None):
        pass

# ============================================================
# 2. TERRENO (superficie y gradiente)
# ============================================================
class Terreno:
    def __init__(self, funcion_altura):
        self.altura = funcion_altura

    def gradiente(self, x, y, h=1e-5):
        dzdx = (self.altura(x+h, y) - self.altura(x-h, y)) / (2*h)
        dzdy = (self.altura(x, y+h) - self.altura(x, y-h)) / (2*h)
        return dzdx, dzdy

    def energia_potencial(self, x, y, masa, g):
        return masa * g * self.altura(x, y)

# ============================================================
# 3. PARTÍCULA (hereda de SistemaFisico)
# ============================================================
class Particula(SistemaFisico):
    def __init__(self, terreno, masa=1.0, rozamiento=0.1, gravedad=9.8):
        self.terreno = terreno
        self.masa = masa
        self.rozamiento = rozamiento
        self.gravedad = gravedad

    def derivada(self, t, estado):
        x, y, vx, vy = estado
        dzdx, dzdy = self.terreno.gradiente(x, y)
        ax = -self.gravedad * dzdx - self.rozamiento * vx
        ay = -self.gravedad * dzdy - self.rozamiento * vy
        return [vx, vy, ax, ay]

    def simular(self, t_span, estado_inicial, t_eval=None):
        sol = solve_ivp(self.derivada, t_span, estado_inicial,
                        t_eval=t_eval, method='RK45', rtol=1e-6)
        return sol.t, sol.y

    def energia_cinetica(self, estado):
        x, y, vx, vy = estado
        return 0.5 * self.masa * (vx**2 + vy**2)

    def energia_potencial(self, estado):
        x, y, vx, vy = estado
        return self.terreno.energia_potencial(x, y, self.masa, self.gravedad)

    def energia_total(self, estado):
        return self.energia_cinetica(estado) + self.energia_potencial(estado)

# ============================================================
# 4. SIMULADOR INTERACTIVO (incorpora análisis y visualización)
# ============================================================
class SimuladorInteractivo:
    def __init__(self):
        # Definir la superficie (montaña con senos y exponencial)
        def montaña(x, y):
            return (np.sin(0.8*x) * np.cos(0.8*y) +
                    0.5 * np.sin(1.5*x) * np.cos(1.2*y) +
                    0.3 * np.exp(-(x**2 + y**2)/4))
        self.terreno = Terreno(montaña)

        # Parámetros por defecto
        self.masa = 1.0
        self.rozamiento = 0.1
        self.gravedad = 9.8
        self.dt_sim = 0.05
        self.t_max = 12.0

        # Estado inicial
        self.posicion = np.array([0.0, 0.0])
        self.velocidad = np.array([0.0, 0.0])

        # Datos de la última simulación
        self.trayectoria = None   # array (N,2) de posiciones
        self.tiempos = None
        self.energias = None      # (N,3) -> Ec, Ep, Etot
        self.particula = None

        # Crear controles
        self.crear_interfaz()

    def crear_interfaz(self):
        # Controles de posición inicial
        self.slider_x = widgets.FloatSlider(value=-2.0, min=-3.0, max=3.0, step=0.1,
                                            description='Pos X:', style={'description_width': 'initial'})
        self.slider_y = widgets.FloatSlider(value=1.8, min=-2.8, max=2.8, step=0.1,
                                            description='Pos Y:', style={'description_width': 'initial'})

        # Controles de velocidad inicial
        self.slider_vx = widgets.FloatSlider(value=-2.0, min=-5.0, max=5.0, step=0.1,
                                             description='Vel X:', style={'description_width': 'initial'})
        self.slider_vy = widgets.FloatSlider(value=-4.0, min=-5.0, max=5.0, step=0.1,
                                             description='Vel Y:', style={'description_width': 'initial'})

        # Controles de parámetros físicos
        self.slider_roz = widgets.FloatSlider(value=0.1, min=0.0, max=1.0, step=0.01,
                                              description='Rozamiento μ:', style={'description_width': 'initial'})
        self.slider_grav = widgets.FloatSlider(value=9.8, min=2.0, max=20.0, step=0.5,
                                               description='Gravedad g:', style={'description_width': 'initial'})
        self.slider_masa = widgets.FloatSlider(value=1.0, min=0.2, max=5.0, step=0.1,
                                               description='Masa (kg):', style={'description_width': 'initial'})

        # Botón ejecutar
        self.btn_simular = widgets.Button(description="▶️ Simular", button_style='success')

        # Áreas de salida
        self.out_graficas = widgets.Output()
        self.out_consulta = widgets.Output()

        # Controles para consulta de energía en un punto
        self.consulta_x = widgets.FloatText(value=0.0, description='x:', step=0.1)
        self.consulta_y = widgets.FloatText(value=0.0, description='y:', step=0.1)
        self.btn_consulta = widgets.Button(description="🔍 Calcular energía en (x,y)")

        # Organizar layout
        controles_iniciales = widgets.HBox([
            widgets.VBox([widgets.HTML("<b>Posición inicial</b>"), self.slider_x, self.slider_y]),
            widgets.VBox([widgets.HTML("<b>Velocidad inicial</b>"), self.slider_vx, self.slider_vy])
        ])
        controles_fisica = widgets.HBox([
            self.slider_roz, self.slider_grav, self.slider_masa
        ])
        panel_consulta = widgets.HBox([self.consulta_x, self.consulta_y, self.btn_consulta])

        self.ui = widgets.VBox([
            controles_iniciales,
            controles_fisica,
            self.btn_simular,
            panel_consulta,
            self.out_consulta,
            self.out_graficas
        ])

        # Conectar eventos
        self.btn_simular.on_click(self.ejecutar_simulacion)
        self.btn_consulta.on_click(self.mostrar_energia_punto)

        display(self.ui)

    def ejecutar_simulacion(self, _):
        with self.out_graficas:
            clear_output(wait=True)
            # Recoger valores actuales
            self.posicion = np.array([self.slider_x.value, self.slider_y.value])
            self.velocidad = np.array([self.slider_vx.value, self.slider_vy.value])
            self.masa = self.slider_masa.value
            self.rozamiento = self.slider_roz.value
            self.gravedad = self.slider_grav.value

            # Crear partícula con estos parámetros
            self.particula = Particula(self.terreno, self.masa, self.rozamiento, self.gravedad)
            estado0 = [self.posicion[0], self.posicion[1], self.velocidad[0], self.velocidad[1]]

            # Simular con SciPy
            t_eval = np.linspace(0, self.t_max, 500)
            self.tiempos, sol = self.particula.simular((0, self.t_max), estado0, t_eval=t_eval)
            # sol[0]=x, sol[1]=y, sol[2]=vx, sol[3]=vy
            self.trayectoria = np.vstack((sol[0], sol[1])).T

            # Calcular energías en cada instante
            Ec = [0.5*self.masa*(vx**2+vy**2) for vx,vy in zip(sol[2], sol[3])]
            Ep = [self.terreno.energia_potencial(x,y,self.masa,self.gravedad) for x,y in zip(sol[0], sol[1])]
            self.energias = np.array([Ec, Ep, np.array(Ec)+np.array(Ep)]).T

            # Generar gráficas
            self.graficar_todo(sol)

    def graficar_todo(self, sol):
        # --- Figura 1: Trayectoria 2D + mapa de colores de altura ---
        fig1, ax1 = plt.subplots(figsize=(7,6))
        x_grid = np.linspace(-3, 3, 80)
        y_grid = np.linspace(-3, 3, 80)
        X, Y = np.meshgrid(x_grid, y_grid)
        Z = self.terreno.altura(X, Y)
        cont = ax1.contourf(X, Y, Z, levels=30, cmap='terrain', alpha=0.8)
        ax1.contour(X, Y, Z, levels=10, colors='black', linewidths=0.3, alpha=0.5)
        ax1.plot(self.trayectoria[:,0], self.trayectoria[:,1], 'r-', linewidth=2, label='Trayectoria')
        ax1.scatter(self.trayectoria[0,0], self.trayectoria[0,1], c='green', s=80, label='Inicio', zorder=5)
        ax1.scatter(self.trayectoria[-1,0], self.trayectoria[-1,1], c='blue', s=80, label='Fin', zorder=5)
        ax1.set_xlim(-3,3); ax1.set_ylim(-3,3); ax1.set_aspect('equal')
        ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.legend()
        plt.colorbar(cont, ax=ax1, label='Altura')
        ax1.set_title('Trayectoria sobre el mapa topográfico')

        # --- Figura 2: Energías a lo largo del tiempo ---
        fig2, ax2 = plt.subplots(figsize=(10,4))
        ax2.plot(self.tiempos, self.energias[:,0], label='Cinética')
        ax2.plot(self.tiempos, self.energias[:,1], label='Potencial')
        ax2.plot(self.tiempos, self.energias[:,2], 'k--', label='Total')
        ax2.set_xlabel('Tiempo (s)'); ax2.set_ylabel('Energía (J)')
        ax2.legend(); ax2.grid(True); ax2.set_title('Evolución de energías')

        # --- Figura 3: Superficie 3D con trayectoria ---
        fig3 = plt.figure(figsize=(8,6))
        ax3 = fig3.add_subplot(111, projection='3d')
        surf = ax3.plot_surface(X, Y, Z, cmap='terrain', alpha=0.7, linewidth=0, antialiased=True)
        # Altura de la trayectoria según el terreno
        z_tray = [self.terreno.altura(x,y) for x,y in self.trayectoria]
        ax3.plot(self.trayectoria[:,0], self.trayectoria[:,1], z_tray, 'r-', linewidth=2)
        ax3.scatter(self.trayectoria[0,0], self.trayectoria[0,1], z_tray[0], c='green', s=50)
        ax3.scatter(self.trayectoria[-1,0], self.trayectoria[-1,1], z_tray[-1], c='blue', s=50)
        ax3.set_xlim(-3,3); ax3.set_ylim(-3,3)
        ax3.set_xlabel('X'); ax3.set_ylabel('Y'); ax3.set_zlabel('Altura')
        ax3.set_title('Trayectoria 3D sobre la superficie')
        fig3.colorbar(surf, ax=ax3, shrink=0.5, aspect=10)

        plt.show()

    def mostrar_energia_punto(self, _):
        with self.out_consulta:
            clear_output(wait=True)
            if self.particula is None:
                print("⚠️ Primero ejecuta una simulación.")
                return
            x = self.consulta_x.value
            y = self.consulta_y.value
            # La energía potencial en ese punto (independiente de la velocidad)
            Ep = self.terreno.energia_potencial(x, y, self.masa, self.gravedad)
            # Para la energía cinética necesitaríamos la velocidad en ese punto.
            # Podemos interpolar la velocidad de la simulación más cercana en el espacio.
            # Pero como es un punto arbitrario, mostramos la potencial y, si está cerca de la trayectoria,
            # podemos aproximar. Para simplificar, mostramos la potencial y la altura.
            altura = self.terreno.altura(x, y)
            print(f"📍 Punto ({x:.2f}, {y:.2f})")
            print(f"   Altura: {altura:.3f} m")
            print(f"   Energía potencial (masa={self.masa:.2f} kg, g={self.gravedad:.1f}): {Ep:.3f} J")
            print("   (La energía cinética depende de la velocidad instantánea; si el punto está sobre la trayectoria simulada, consulta la gráfica de energías en el tiempo correspondiente.)")

# ============================================================
# 5. ANÁLISIS OPCIONAL (curve_fit) - Se puede activar como extra
# ============================================================
def analisis_avanzado():
    """Función separada para ajuste de parámetros (cumple SciPy curve_fit)."""
    # Definir terreno igual
    def montaña(x,y):
        return (np.sin(0.8*x)*np.cos(0.8*y) + 0.5*np.sin(1.5*x)*np.cos(1.2*y) + 0.3*np.exp(-(x**2+y**2)/4))
    terreno = Terreno(montaña)
    # Parámetros reales
    roz_real = 0.12
    g_real = 9.8
    masa = 1.0
    estado0 = [1.0, 1.5, 0.0, 0.0]
    t_max = 10.0
    t_med = np.linspace(0, t_max, 150)
    particula_real = Particula(terreno, masa, roz_real, g_real)
    _, y_real = particula_real.simular((0, t_max), estado0, t_eval=t_med)
    # Añadir ruido
    ruido_std = 0.03
    x_med = y_real[0] + np.random.normal(0, ruido_std, size=t_med.shape)
    y_med = y_real[1] + np.random.normal(0, ruido_std, size=t_med.shape)

    def modelo_ajuste(t, roz, g):
        part_temp = Particula(terreno, masa, roz, g)
        _, y_temp = part_temp.simular((0, t_max), estado0, t_eval=t)
        return np.concatenate([y_temp[0], y_temp[1]])

    ydata = np.concatenate([x_med, y_med])
    popt, pcov = curve_fit(modelo_ajuste, t_med, ydata, p0=[0.2, 9.5], maxfev=5000)
    print("=== ANÁLISIS AVANZADO (curve_fit) ===")
    print(f"Rozamiento real: {roz_real:.3f} -> ajustado: {popt[0]:.3f} ± {np.sqrt(pcov[0,0]):.4f}")
    print(f"Gravedad real: {g_real:.1f} -> ajustada: {popt[1]:.2f} ± {np.sqrt(pcov[1,1]):.2f}")

# ============================================================
# 6. EJECUCIÓN PRINCIPAL
# ============================================================
if __name__ == "__main__":
    print("🎯 SIMULADOR INTERACTIVO DE CANICA SOBRE MONTAÑA")
    print("Ajusta los parámetros y haz clic en 'Simular'")
    print("Para el análisis de ajuste de parámetros (curve_fit), ejecuta la función analisis_avanzado()")
    sim = SimuladorInteractivo()
    # Si quieres demostrar el curve_fit, descomenta la siguiente línea:
    # analisis_avanzado()

🎯 SIMULADOR INTERACTIVO DE CANICA SOBRE MONTAÑA
Ajusta los parámetros y haz clic en 'Simular'
Para el análisis de ajuste de parámetros (curve_fit), ejecuta la función analisis_avanzado()
